# Deploy Clinical Data Agent

This notebook builds the Genie Agent configuration from the repository JSON files and safely applies it to the existing Clinical Data Agent.

Deployment workflow:

1. Build and validate the repository configuration
2. Retrieve the existing Genie Agent
3. Back up the current Agent configuration
4. Confirm the Space ID and Agent title
5. Perform a dry run
6. Deploy only after validation succeeds
7. Validate the deployed configuration

In [0]:
from pathlib import Path
import sys


# Find the repository root
current_path = Path.cwd().resolve()

repo_root = None

for candidate in [current_path] + list(current_path.parents):
    if (
        (candidate / "config").exists()
        and (candidate / "src" / "deployment").exists()
    ):
        repo_root = candidate
        break


if repo_root is None:
    raise RuntimeError(
        "Could not locate the repository root."
    )


deployment_src = repo_root / "src" / "deployment"

if str(deployment_src) not in sys.path:
    sys.path.insert(
        0,
        str(deployment_src)
    )


print(f"Repository root: {repo_root}")
print(f"Deployment source: {deployment_src}")

In [0]:
# Existing Genie Agent to update
SPACE_ID = "01f1b4aab3d7116dbd4f02d504887741"

# Safety check
EXPECTED_TITLE = "Clinical Data Agent"

In [0]:
import importlib

import build_genie_config
import deploy_genie_agent


# Reload updated repository modules
importlib.reload(build_genie_config)
importlib.reload(deploy_genie_agent)


from build_genie_config import (
    build_genie_config,
    export_genie_config
)

from deploy_genie_agent import (
    deploy_genie_agent,
    validate_deployed_agent,
    get_genie_space
)


# Build and export a readable preview
config = build_genie_config()
output_file = export_genie_config()


sql_snippets = (
    config
    .get("instructions", {})
    .get("sql_snippets", {})
)


print("Genie configuration built successfully.")
print()
print(f"Version: {config['version']}")
print(f"Tables: {len(config['data_sources']['tables'])}")
print(f"Sample questions: {len(config['config']['sample_questions'])}")
print(f"SQL examples: {len(config['instructions']['example_question_sqls'])}")
print(f"Join specs: {len(config['instructions']['join_specs'])}")
print(f"SQL filters: {len(sql_snippets.get('filters', []))}")
print(f"SQL expressions: {len(sql_snippets.get('expressions', []))}")
print(f"SQL measures: {len(sql_snippets.get('measures', []))}")
print(f"Benchmarks: {len(config['benchmarks']['questions'])}")
print()
print(f"Generated preview: {output_file}")

In [0]:
dry_run_result = deploy_genie_agent(
    space_id=SPACE_ID,
    expected_title=EXPECTED_TITLE,
    dry_run=True
)

dry_run_result

In [0]:
import json

from deploy_genie_agent import get_genie_space
from build_genie_config import build_genie_config


# Load current Agent configuration
current_space = get_genie_space(SPACE_ID)

current_serialized = json.loads(
    current_space["serialized_space"]
)

# Load repository-built configuration
new_config = build_genie_config()


def show_structure(name, config):
    print(f"\n{name}")
    print("=" * len(name))

    print("Top-level keys:")
    print(sorted(config.keys()))

    print("\nconfig keys:")
    print(sorted(config.get("config", {}).keys()))

    print("\ndata_sources keys:")
    print(sorted(config.get("data_sources", {}).keys()))

    print("\ninstructions keys:")
    print(sorted(config.get("instructions", {}).keys()))

    print("\nbenchmark keys:")
    print(sorted(config.get("benchmarks", {}).keys()))


show_structure(
    "CURRENT GENIE AGENT",
    current_serialized
)

show_structure(
    "NEW REPOSITORY CONFIG",
    new_config
)

In [0]:
# Deploy the repository configuration to the existing Genie Agent
deployment_result = deploy_genie_agent(
    space_id=SPACE_ID,
    expected_title=EXPECTED_TITLE,
    dry_run=False
)

deployment_result

In [0]:
# Validate the deployed Genie configuration
validation_summary = validate_deployed_agent(
    SPACE_ID
)

validation_summary